# Training & Evaluation with Data Augmentation

In [ ]:
import torch
import torchmetrics
import numpy as np
import h5py
import warnings
import torchvision.transforms.functional as F
from tqdm.notebook import tqdm

from dlmi.utils import set_seed, get_device, save_submission
from dlmi.dataset import H5Dataset, get_dataloader
from dlmi.model import get_finetunable_dinov2
from dlmi.transforms import get_ood_transform
from dlmi.train import train
from dlmi.test import tta_predict, evaluate_no_tta, evaluate_with_tta

warnings.filterwarnings("ignore", category=UserWarning)

## Configuration

In [ ]:
# Paths
TRAIN_PATH = "../data/train.h5"
VAL_PATH = "../data/val.h5"
TEST_PATH = "../data/test.h5"

# Hyperparameters
SEED = 0
BATCH_SIZE = 16
LR = 0.001
NUM_EPOCHS = 100
PATIENCE = 10
IMG_SIZE = 98

# Key parameters
MODEL_NAME = (
    "dinov2_vits14"  # Choose from "dinov2_vits14", "dinov2_vitb14", "dinov2_vitl14"
)
NB_LAYERS_TO_FINE_TUNE = 2

MODEL_SAVE_PATH = (
    f"../models/augmented_{MODEL_NAME}_{NB_LAYERS_TO_FINE_TUNE}_layers.pth"
)
SUBMISSION_PATH = (
    f"../results/submission_augmented_{MODEL_NAME}_{NB_LAYERS_TO_FINE_TUNE}_layers.csv"
)

In [ ]:
set_seed(SEED)
device = get_device()
print(f"Device: {device}")

## 1. Prepare datasets and precompute features

In [ ]:
train_preprocessing = get_ood_transform(size=IMG_SIZE, train=True)
val_preprocessing = get_ood_transform(size=IMG_SIZE, train=False)

train_ds = H5Dataset(TRAIN_PATH, transform=train_preprocessing, mode="train")
val_ds = H5Dataset(VAL_PATH, transform=val_preprocessing, mode="train")

train_loader = get_dataloader(train_ds, batch_size=BATCH_SIZE, shuffle=True)
val_loader = get_dataloader(val_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train: {len(train_ds)} samples, Val: {len(val_ds)} samples")

## 2. Train the linear probe

In [ ]:
model = get_finetunable_dinov2(
    MODEL_NAME, num_blocks_to_unfreeze=NB_LAYERS_TO_FINE_TUNE, device=device
)
criterion = torch.nn.BCELoss()
metric = torchmetrics.Accuracy("binary")

optimizer = torch.optim.Adam(
    [
        {"params": model.backbone.parameters(), "lr": 1e-5},  # small LR for backbone
        {"params": model.head.parameters(), "lr": LR},  # higher LR for head
    ]
)

In [ ]:
history = train(
    model,
    train_loader,
    val_loader,
    optimizer,
    criterion,
    metric,
    device,
    num_epochs=NUM_EPOCHS,
    patience=PATIENCE,
    save_path=MODEL_SAVE_PATH,
)

### Training curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(history["train_loss"], label="Train")
axes[0].plot(history["val_loss"], label="Val")
axes[0].set_title("Loss")
axes[0].set_xlabel("Epoch")
axes[0].legend()

axes[1].plot(history["train_metric"], label="Train")
axes[1].plot(history["val_metric"], label="Val")
axes[1].set_title("Accuracy")
axes[1].set_xlabel("Epoch")
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Evaluate on the validation set

In [ ]:
model = get_finetunable_dinov2(
    MODEL_NAME, num_blocks_to_unfreeze=NB_LAYERS_TO_FINE_TUNE, device=device
)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))

acc = evaluate_no_tta(model, val_loader, device)
print(f"Val Accuracy (no TTA): {acc:.4f}")

In [ ]:
model = get_finetunable_dinov2(
    MODEL_NAME, num_blocks_to_unfreeze=NB_LAYERS_TO_FINE_TUNE, device=device
)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))

acc = evaluate_with_tta(model, VAL_PATH, val_preprocessing, device)
print(f"Val Accuracy with TTA: {acc:.4f}")

## 4. Generate test predictions

In [ ]:
model = get_finetunable_dinov2(
    MODEL_NAME, num_blocks_to_unfreeze=NB_LAYERS_TO_FINE_TUNE, device=device
)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))
model.eval()

test_ds = H5Dataset(TEST_PATH, transform=val_preprocessing, mode="test")
test_loader = get_dataloader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

test_preds_no_tta = []
with torch.no_grad():
    for imgs, _ in tqdm(test_loader, desc="Test (no TTA)"):
        imgs = imgs.to(device)
        preds = model(imgs)
        test_preds_no_tta.append(preds.cpu())

test_preds_no_tta = torch.cat(test_preds_no_tta).squeeze().tolist()
test_ids_no_tta = [int(i) for i in test_ds.image_ids]

submission_no_tta = save_submission(
    test_ids_no_tta, test_preds_no_tta, SUBMISSION_PATH.replace(".csv", "_no_tta.csv")
)
print(f"Submission (no TTA) saved ({len(submission_no_tta)} rows)")
submission_no_tta.head()

In [ ]:
model = get_finetunable_dinov2(
    MODEL_NAME, num_blocks_to_unfreeze=NB_LAYERS_TO_FINE_TUNE, device=device
)
model.load_state_dict(torch.load(MODEL_SAVE_PATH, weights_only=True))
model.eval()

test_ids, test_preds = [], []

with h5py.File(TEST_PATH, "r") as hdf:
    for test_id in tqdm(hdf.keys(), desc="Test TTA"):
        img = val_preprocessing(torch.tensor(np.array(hdf[test_id]["img"])).float())
        pred = tta_predict(model, img, device)
        test_ids.append(int(test_id))
        test_preds.append(pred)

submission = save_submission(
    test_ids, test_preds, SUBMISSION_PATH.replace(".csv", "_tta.csv")
)
print(f"Submission saved to {SUBMISSION_PATH} ({len(submission)} rows)")
submission.head()